In [ ]:
!python --version

Python 3.13.18


In [ ]:
!pip install apache-airflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 82.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 737.4/737.4 kB 27.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 24.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 75.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 778.7/778.7 kB 28.7 MB/s  0:00:00
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions-0.63b1


In [ ]:
!airflow version

3.4.0


In [ ]:
from airflow.sdk import dag, task
from datetime import datetime

@dag(
    dag_id="hr_recruitment_analytics_elt",
    schedule=None,
    start_date=datetime(2026, 1, 1),
    catchup=False,
    tags=["HR Analytics", "PySpark", "ELT"]
)
def hr_recruitment_analytics_elt():

    @task
    def extract_recruitment_data():
        print("Step 1: Recruitment data extracted successfully.")
        return "/content/recruitment_curated.csv"

    @task
    def transform_with_pyspark(input_file):
        print("Step 2: PySpark transformation initiated.")
        print(f"Source file: {input_file}")
        return "/content/recruitment_pyspark_final.csv"

    @task
    def load_analytics_dataset(output_file):
        print("Step 3: Final analytics dataset prepared.")
        print(f"Output file: {output_file}")

    source = extract_recruitment_data()
    transformed = transform_with_pyspark(source)
    load_analytics_dataset(transformed)

hr_recruitment_dag = hr_recruitment_analytics_elt()

In [ ]:
!airflow dags test hr_recruitment_analytics_elt 2026-11-08


Please confirm database initialize (or wait 4 seconds to skip it). Are you sure? [y/N]
2026-11-08T16:38:05.837101Z [info     ] Log template table does not exist (added in 2.3.0); skipping log template sync. [airflow.utils.db] loc=db.py:1025
2026-11-08T16:38:05.838685Z [info     ] DAG bundles loaded: dags-folder, example_dags, apache-airflow-providers-common-sql-example-dags, apache-airflow-providers-standard-example-dags [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:344
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/sqlalchemy/engine/base.py", line 1977, in _exec_single_context
    self.dialect.do_execute(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        cursor, str_statement, effective_parameters, context
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/sqlalchemy/engine/default.py", line 1192, in do_execute
    cursor.execute(statement, parameters)
    ~~~~~~~~

In [ ]:
!airflow db migrate

2026-11-08T16:39:37.088214Z [info     ] Performing upgrade to the metadata database [airflow.cli.commands.db_command] loc=db_command.py:134 url=sqlite:////root/airflow/airflow.db
2026-11-08T16:39:38.347002Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-11-08T16:39:38.347257Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-11-08T16:39:39.608345Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-11-08T16:39:39.608565Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-11-08T16:39:39.609082Z [info     ] Creating Airflow database tables from the ORM [airflow.utils.db] loc=db.py:750
2026-11-08T16:39:39.609310Z [info     ] Creating global lock context   [airflow.utils.db] loc=db.py:731
2026-11-08T16:39:39.609495Z [info     ] Pool status: Pool size: 5  Connections in pool: 0 Current Overflow: -4

In [ ]:
!airflow dags test hr_recruitment_analytics_elt 2026-11-08

2026-11-08T16:41:26.625532Z [info     ] DAG bundles loaded: dags-folder, example_dags, apache-airflow-providers-common-sql-example-dags, apache-airflow-providers-standard-example-dags [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:344
2026-11-08T16:41:26.822561Z [info     ] Added new DAG bundle dags-folder to the database [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:418
2026-11-08T16:41:26.823035Z [info     ] Added new DAG bundle example_dags to the database [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:418
2026-11-08T16:41:26.823347Z [info     ] Added new DAG bundle apache-airflow-providers-common-sql-example-dags to the database [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:418
2026-11-08T16:41:26.823635Z [info     ] Added new DAG bundle apache-airflow-providers-standard-example-dags to the database [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:4

In [ ]:
!airflow config get-value core dags_folder

/root/airflow/dags


In [ ]:
import pyspark
print(pyspark.__version__)

4.0.5


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HR_Recruitment_Analytics") \
    .getOrCreate()

print("Spark Version:", spark.version)

2026-11-08T17:16:57.025657Z [info     ] NumExpr defaulting to 2 threads. [numexpr.utils] loc=utils.py:164
Spark Version: 4.0.5


In [ ]:
import os

os.makedirs("/root/airflow/dags", exist_ok=True)

dag_code = '''
from airflow.sdk import dag, task
from datetime import datetime

@dag(
    dag_id="hr_recruitment_analytics_elt",
    schedule=None,
    start_date=datetime(2026, 1, 1),
    catchup=False,
    tags=["HR Analytics", "ELT", "PySpark"]
)
def hr_recruitment_analytics_elt():

    @task
    def extract_recruitment_data():
        print("Step 1: Recruitment data extracted.")
        return "/content/recruitment_curated.csv"

    @task
    def transform_with_pyspark(source_file):
        print("Step 2: PySpark transformation completed.")
        print(f"Source file: {source_file}")
        return "/content/recruitment_pyspark_final.csv"

    @task
    def load_analytics_dataset(output_file):
        print("Step 3: Final analytics dataset prepared.")
        print(f"Output file: {output_file}")

    source = extract_recruitment_data()
    transformed = transform_with_pyspark(source)
    load_analytics_dataset(transformed)

hr_recruitment_analytics_elt()
'''

with open("/root/airflow/dags/hr_recruitment_analytics_elt.py", "w") as f:
    f.write(dag_code)

print("Airflow DAG file created successfully.")

Airflow DAG file created successfully.


In [ ]:
!airflow dags list | grep hr_recruitment

In [ ]:
!airflow dags list-import-errors

No data found


In [ ]:
!airflow dags list --subdir /root/airflow/dags/hr_recruitment_analytics_elt.py

Usage: airflow [-h] GROUP_OR_COMMAND ...

Positional Arguments:
  GROUP_OR_COMMAND

    Groups
      assets            Manage assets
      backfill          Manage backfills
      config            View configuration
      connections       Manage connections
      dags              Manage DAGs
      db                Database operations
      db-manager        Manage externally connected database managers
      jobs              Manage jobs
      partitions        Manage Dag run partition metadata
      pools             Manage pools
      providers         Display providers
      state-store       Manage task and asset state storage
      tasks             Manage tasks
      teams             Manage teams
      variables         Manage variables

    Commands:
      api-server        Start an Airflow API server instance
      cheat-sheet       Display cheat sheet
      dag-processor     Start a dag processor instance
      info              Show information about current Airflow and 

In [ ]:
!ls -l /root/airflow/dags/hr_recruitment_analytics_elt.py

-rw-r--r-- 1 root root 960 Nov  8 17:18 /root/airflow/dags/hr_recruitment_analytics_elt.py


In [ ]:
!airflow dags list

              |              |         |           |             | bundle_versio
dag_id        | fileloc      | owners  | is_paused | bundle_name | n            
==============+==============+=========+===========+=============+==============
aggregate_reg | /usr/local/l | airflow | True      | example_dag | None         
ional_sales   | ib/python3.1 |         |           | s           |              
              | 3/dist-packa |         |           |             |              
              | ges/airflow/ |         |           |             |              
              | example_dags |         |           |             |              
              | /example_ass |         |           |             |              
              | et_partition |         |           |             |              
              | .py          |         |           |             |              
asset1_produc | /usr/local/l | airflow | True      | example_dag | None         
er            | ib/python3.1

In [ ]:
!airflow dags list | grep hr_recruitment_analytics_elt

In [ ]:
hr_recruitment_analytics_elt

<function __main__.hr_recruitment_analytics_elt()>

In [ ]:
!airflow dags list | grep hr_recruitment

In [ ]:
!airflow dag-processor --help

Usage: airflow dag-processor [-h] [-B BUNDLE_NAME] [-D] [-d] [-l LOG_FILE]
                             [-n NUM_RUNS] [--pid [PID]] [--stderr STDERR]
                             [--stdout STDOUT] [-v]

Start a dag processor instance

Options:
  -h, --help            show this help message and exit
  -B, --bundle-name BUNDLE_NAME
                        The name of the DAG bundle to use; may be provided more than once
  -D, --daemon          Daemonize instead of running in the foreground
  -d, --dev             Start in development mode with hot-reload enabled
  -l, --log-file LOG_FILE
                        Location of the log file
  -n, --num-runs NUM_RUNS
                        Set the number of runs to execute before exiting
  --pid [PID]           PID file location
  --stderr STDERR       Redirect stderr to this file
  --stdout STDOUT       Redirect stdout to this file
  -v, --verbose         Make logging output more verbose


In [ ]:
!airflow dag-processor --num-runs 1

2026-11-08T17:33:08.184342Z [info     ] Starting the Dag Processor Job [airflow.jobs.dag_processor_job_runner.DagProcessorJobRunner] loc=dag_processor_job_runner.py:59
2026-11-08T17:33:08.184941Z [info     ] Processing files using up to 2 processes at a time  [airflow.dag_processing.manager.DagFileProcessorManager] loc=manager.py:364
2026-11-08T17:33:08.185078Z [info     ] Process each file at most once every 30 seconds [airflow.dag_processing.manager.DagFileProcessorManager] loc=manager.py:365
2026-11-08T17:33:08.185643Z [info     ] DAG bundles loaded: dags-folder, example_dags, apache-airflow-providers-common-sql-example-dags, apache-airflow-providers-standard-example-dags [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:344
2026-11-08T17:33:08.194507Z [info     ] DAG bundles loaded: dags-folder, example_dags, apache-airflow-providers-common-sql-example-dags, apache-airflow-providers-standard-example-dags [airflow.dag_processing.bundles.manager.DagBundlesMana

In [ ]:
!airflow dags list | grep hr_recruitment

hr_recruitment_analytics_elt                           | /root/airflow/dags/hr_recruitment_analytics_elt.py    | airflow | True      | dags-folder                                      | None          


In [ ]:
!airflow dags test hr_recruitment_analytics_elt 2026-11-08

2026-11-08T17:37:21.266813Z [info     ] DAG bundles loaded: dags-folder, example_dags, apache-airflow-providers-common-sql-example-dags, apache-airflow-providers-standard-example-dags [airflow.dag_processing.bundles.manager.DagBundlesManager] loc=manager.py:344
2026-11-08T17:37:21.513077Z [info     ] Filling up the DagBag from /root/airflow/dags [airflow.dag_processing.dagbag.BundleDagBag] loc=dagbag.py:583
2026-11-08T17:37:21.572975Z [info     ] Bulk-writing dags to db        [airflow.serialization.definitions.dag] count=1 loc=dag.py:247
2026-11-08T17:37:21.600045Z [info     ] get next_dagrun_info_v2        [airflow.serialization.definitions.dag] last_automated_run_info=None loc=dag.py:491 next_info=None
2026-11-08T17:37:21.600357Z [info     ] setting next dagrun info       [airflow.models.dag] loc=dag.py:887 next_dagrun=None next_dagrun_create_after=None next_dagrun_data_interval_end=None next_dagrun_data_interval_start=None next_dagrun_partition_date=None next_dagrun_partition_key=N

In [ ]:
!airflow version

3.4.0


In [ ]:
import os

print("Final file exists:", os.path.exists("/content/recruitment_pyspark_final.csv"))

Final file exists: False


In [ ]:
!ls -lh /content | grep recruitment

In [ ]:
df = spark.read.csv(
    "/content/recruitment_curated.csv",
    header=True,
    inferSchema=True
)

print("Total Records:", df.count())

Total Records: 500


In [ ]:
!ls -lh /content

total 4.0K
drwxr-xr-x 1 root root 4.0K Nov  6 16:29 sample_data


In [ ]:
df.printSchema()

root
 |-- candidate_id: integer (nullable = true)
 |-- candidate_name: string (nullable = true)
 |-- company: string (nullable = true)
 |-- job_role: string (nullable = true)
 |-- round_stage: string (nullable = true)
 |-- result: string (nullable = true)
 |-- completion_status: string (nullable = true)
 |-- validation_status: string (nullable = true)
 |-- interview_date: date (nullable = true)
 |-- location: string (nullable = true)
 |-- source: string (nullable = true)
 |-- experience_years: integer (nullable = true)
 |-- candidate_outcome: string (nullable = true)



In [ ]:
!pwd

/content


In [ ]:
!ls -lh /content/recruitment*.csv

ls: cannot access '/content/recruitment*.csv': No such file or directory


In [ ]:
!find / -name "recruitment*.csv" 2>/dev/null

/recruitment_curated.csv
/recruitment_pyspark_final.csv


In [ ]:
import os

print("Source file:", os.path.exists("/recruitment_curated.csv"))
print("Final file:", os.path.exists("/recruitment_pyspark_final.csv"))

Source file: True
Final file: True


In [ ]:
!airflow dags list-runs -d hr_recruitment_analytics_elt

Usage: airflow [-h] GROUP_OR_COMMAND ...

Positional Arguments:
  GROUP_OR_COMMAND

    Groups
      assets            Manage assets
      backfill          Manage backfills
      config            View configuration
      connections       Manage connections
      dags              Manage DAGs
      db                Database operations
      db-manager        Manage externally connected database managers
      jobs              Manage jobs
      partitions        Manage Dag run partition metadata
      pools             Manage pools
      providers         Display providers
      state-store       Manage task and asset state storage
      tasks             Manage tasks
      teams             Manage teams
      variables         Manage variables

    Commands:
      api-server        Start an Airflow API server instance
      cheat-sheet       Display cheat sheet
      dag-processor     Start a dag processor instance
      info              Show information about current Airflow and 

In [ ]:
!airflow dags list-runs --help

Usage: airflow dags list-runs [-h] [-e END_DATE] [--limit LIMIT]
                              [--no-backfill] [-o (table, json, yaml, plain)]
                              [-s START_DATE]
                              [--state queued, running, success, failed] [-v]
                              dag_id

List DAG runs given a DAG id. If state option is given, it will only search for all the dagruns with the given state. If no_backfill option is given, it will filter out all backfill dagruns for given dag id. If start_date is given, it will filter out all the dagruns that were executed before this date. If end_date is given, it will filter out all the dagruns that were executed after this date. If limit is given, it will return only the most recent N runs after filters and sorting are applied.

Positional Arguments:
  dag_id                The id of the dag

Options:
  -h, --help            show this help message and exit
  -e, --end-date END_DATE
                        Override end_dat

In [ ]:
!airflow dags list-runs hr_recruitment_analytics_elt

           |           |         |           | logical_d | start_dat |          
dag_id     | run_id    | state   | run_after | ate       | e         | end_date 
===========+===========+=========+===========+===========+===========+==========
hr_recruit | manual__2 | success | 2026-11-0 | 2026-11-0 | 2026-11-0 | 2026-11-0
ment_analy | 026-11-08 |         | 8T17:37:2 | 8T00:00:0 | 8T17:37:2 | 8T17:37:3
tics_elt   | T17:37:22 |         | 2.242830+ | 0+00:00   | 2.281379+ | 0.256801+
           | .242830+0 |         | 00:00     |           | 00:00     | 00:00    
           | 0:00      |         |           |           |           |          
                                                                                


In [ ]:
df.show(10, truncate=False)

+------------+--------------+-----------+------------------+---------------+------+-----------------+-----------------+--------------+---------+-------------+----------------+-----------------+
|candidate_id|candidate_name|company    |job_role          |round_stage    |result|completion_status|validation_status|interview_date|location |source       |experience_years|candidate_outcome|
+------------+--------------+-----------+------------------+---------------+------+-----------------+-----------------+--------------+---------+-------------+----------------+-----------------+
|1           |Candidate 001 |NovaTech   |Power BI Developer|Screening      |Fail  |Complete         |Validated        |2026-03-13    |Bengaluru|Agency       |10              |Not Selected     |
|2           |Candidate 002 |DataBridge |Business Analyst  |Screening      |Pass  |Complete         |Validated        |2026-01-14    |Mumbai   |Naukri       |6               |Selected         |
|3           |Candidate 003 |I

In [ ]:
from pyspark.sql import functions as F

job_role_analysis = (
    df.groupBy("job_role")
      .agg(
          F.count("*").alias("total_candidates"),
          F.sum(
              F.when(F.col("candidate_outcome") == "Selected", 1).otherwise(0)
          ).alias("selected_candidates")
      )
      .withColumn(
          "selection_rate",
          F.round(
              F.col("selected_candidates") * 100.0 / F.col("total_candidates"),
              2
          )
      )
      .orderBy(F.desc("selection_rate"))
)

job_role_analysis.show()

+------------------+----------------+-------------------+--------------+
|          job_role|total_candidates|selected_candidates|selection_rate|
+------------------+----------------+-------------------+--------------+
|     Data Engineer|             102|                 52|         50.98|
|      Data Analyst|              96|                 39|         40.63|
|Power BI Developer|              99|                 39|         39.39|
|  Business Analyst|             109|                 42|         38.53|
|        HR Analyst|              94|                 35|         37.23|
+------------------+----------------+-------------------+--------------+



In [ ]:
source_analysis = (
    df.groupBy("source")
      .agg(
          F.count("*").alias("total_candidates"),
          F.sum(
              F.when(F.col("candidate_outcome") == "Selected", 1).otherwise(0)
          ).alias("selected_candidates")
      )
      .withColumn(
          "selection_rate",
          F.round(
              F.col("selected_candidates") * 100.0 / F.col("total_candidates"),
              2
          )
      )
      .orderBy(F.desc("selection_rate"))
)

source_analysis.show()

+-------------+----------------+-------------------+--------------+
|       source|total_candidates|selected_candidates|selection_rate|
+-------------+----------------+-------------------+--------------+
|       Naukri|              98|                 47|         47.96|
|     LinkedIn|             100|                 46|          46.0|
|Career Portal|             104|                 43|         41.35|
|     Referral|             101|                 37|         36.63|
|       Agency|              97|                 34|         35.05|
+-------------+----------------+-------------------+--------------+



In [ ]:
location_analysis = (
    df.groupBy("location")
      .agg(
          F.count("*").alias("total_candidates"),
          F.sum(
              F.when(F.col("candidate_outcome") == "Selected", 1).otherwise(0)
          ).alias("selected_candidates")
      )
      .withColumn(
          "selection_rate",
          F.round(
              F.col("selected_candidates") * 100.0 / F.col("total_candidates"),
              2
          )
      )
      .orderBy(F.desc("selection_rate"))
)

location_analysis.show()

+---------+----------------+-------------------+--------------+
| location|total_candidates|selected_candidates|selection_rate|
+---------+----------------+-------------------+--------------+
|   Mumbai|             102|                 47|         46.08|
|Hyderabad|             112|                 46|         41.07|
|     Pune|             107|                 43|         40.19|
|Bengaluru|              95|                 38|          40.0|
|  Chennai|              84|                 33|         39.29|
+---------+----------------+-------------------+--------------+



In [ ]:
df_transformed = (
    df.withColumn(
        "experience_band",
        F.when(F.col("experience_years") <= 1, "Fresher")
         .when(F.col("experience_years") <= 3, "Junior")
         .when(F.col("experience_years") <= 6, "Mid-Level")
         .otherwise("Experienced")
    )
)

df_transformed.select(
    "candidate_id",
    "job_role",
    "experience_years",
    "experience_band",
    "candidate_outcome"
).show(20, truncate=False)

+------------+------------------+----------------+---------------+-----------------+
|candidate_id|job_role          |experience_years|experience_band|candidate_outcome|
+------------+------------------+----------------+---------------+-----------------+
|1           |Power BI Developer|10              |Experienced    |Not Selected     |
|2           |Business Analyst  |6               |Mid-Level      |Selected         |
|3           |Data Engineer     |3               |Junior         |Selected         |
|4           |Data Engineer     |3               |Junior         |Selected         |
|5           |Power BI Developer|0               |Fresher        |Not Selected     |
|6           |Data Analyst      |5               |Mid-Level      |Not Selected     |
|7           |Business Analyst  |6               |Mid-Level      |Not Selected     |
|8           |Data Engineer     |9               |Experienced    |Not Selected     |
|9           |HR Analyst        |10              |Experienced    

In [ ]:
experience_analysis = (
    df_transformed.groupBy("experience_band")
      .agg(
          F.count("*").alias("total_candidates"),
          F.sum(
              F.when(F.col("candidate_outcome") == "Selected", 1).otherwise(0)
          ).alias("selected_candidates")
      )
      .withColumn(
          "selection_rate",
          F.round(
              F.col("selected_candidates") * 100.0 / F.col("total_candidates"),
              2
          )
      )
      .orderBy(F.desc("selection_rate"))
)

experience_analysis.show()

+---------------+----------------+-------------------+--------------+
|experience_band|total_candidates|selected_candidates|selection_rate|
+---------------+----------------+-------------------+--------------+
|      Mid-Level|             116|                 54|         46.55|
|    Experienced|             200|                 88|          44.0|
|        Fresher|              90|                 37|         41.11|
|         Junior|              94|                 28|         29.79|
+---------------+----------------+-------------------+--------------+



In [ ]:
df_transformed = (
    df_transformed
    .withColumn("interview_year", F.year("interview_date"))
    .withColumn("interview_month", F.month("interview_date"))
)

df_transformed.select(
    "candidate_id",
    "interview_date",
    "interview_year",
    "interview_month"
).show(10)

+------------+--------------+--------------+---------------+
|candidate_id|interview_date|interview_year|interview_month|
+------------+--------------+--------------+---------------+
|           1|    2026-03-13|          2026|              3|
|           2|    2026-01-14|          2026|              1|
|           3|    2026-03-23|          2026|              3|
|           4|    2026-02-19|          2026|              2|
|           5|    2026-07-13|          2026|              7|
|           6|    2026-04-27|          2026|              4|
|           7|    2026-07-09|          2026|              7|
|           8|    2026-08-25|          2026|              8|
|           9|    2026-01-17|          2026|              1|
|          10|    2026-04-19|          2026|              4|
+------------+--------------+--------------+---------------+
only showing top 10 rows


In [ ]:
import pyspark
print("PySpark version:", pyspark.__version__)

PySpark version: 4.0.5


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HR_Recruitment_Analytics") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.5


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HR_Recruitment_Analytics") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.5


In [ ]:
final_df = spark.read.csv(
    "/content/recruitment_pyspark_final.csv",
    header=True,
    inferSchema=True
)

print("Final Records:", final_df.count())

Final Records: 500


In [ ]:
final_df.printSchema()

root
 |-- candidate_id: integer (nullable = true)
 |-- candidate_name: string (nullable = true)
 |-- company: string (nullable = true)
 |-- job_role: string (nullable = true)
 |-- round_stage: string (nullable = true)
 |-- result: string (nullable = true)
 |-- completion_status: string (nullable = true)
 |-- validation_status: string (nullable = true)
 |-- interview_date: date (nullable = true)
 |-- location: string (nullable = true)
 |-- source: string (nullable = true)
 |-- experience_years: integer (nullable = true)
 |-- candidate_outcome: string (nullable = true)
 |-- experience_band: string (nullable = true)
 |-- interview_year: integer (nullable = true)
 |-- interview_month: integer (nullable = true)



In [ ]:
from pyspark.sql import functions as F

final_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in final_df.columns
]).show(truncate=False)

+------------+--------------+-------+--------+-----------+------+-----------------+-----------------+--------------+--------+------+----------------+-----------------+---------------+--------------+---------------+
|candidate_id|candidate_name|company|job_role|round_stage|result|completion_status|validation_status|interview_date|location|source|experience_years|candidate_outcome|experience_band|interview_year|interview_month|
+------------+--------------+-------+--------+-----------+------+-----------------+-----------------+--------------+--------+------+----------------+-----------------+---------------+--------------+---------------+
|0           |0             |0      |0       |0          |0     |0                |0                |0             |0       |0     |0               |0                |0              |0             |0              |
+------------+--------------+-------+--------+-----------+------+-----------------+-----------------+--------------+--------+------+--------

In [ ]:
from pyspark.sql import functions as F

final_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in final_df.columns
]).show(truncate=False)

+------------+--------------+-------+--------+-----------+------+-----------------+-----------------+--------------+--------+------+----------------+-----------------+---------------+--------------+---------------+
|candidate_id|candidate_name|company|job_role|round_stage|result|completion_status|validation_status|interview_date|location|source|experience_years|candidate_outcome|experience_band|interview_year|interview_month|
+------------+--------------+-------+--------+-----------+------+-----------------+-----------------+--------------+--------+------+----------------+-----------------+---------------+--------------+---------------+
|0           |0             |0      |0       |0          |0     |0                |0                |0             |0       |0     |0               |0                |0              |0             |0              |
+------------+--------------+-------+--------+-----------+------+-----------------+-----------------+--------------+--------+------+--------

In [ ]:
duplicate_ids = (
    final_df.groupBy("candidate_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate Candidate IDs:", duplicate_ids.count())

Duplicate Candidate IDs: 0


In [ ]:
duplicate_ids = (
    final_df.groupBy("candidate_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate Candidate IDs:", duplicate_ids.count())

Duplicate Candidate IDs: 0


In [ ]:
final_df.groupBy("candidate_outcome") \
    .count() \
    .orderBy("candidate_outcome") \
    .show()

+-----------------+-----+
|candidate_outcome|count|
+-----------------+-----+
|     Not Selected|  211|
|          Pending|   82|
|         Selected|  207|
+-----------------+-----+



In [ ]:
final_df.groupBy("experience_band") \
    .count() \
    .orderBy("experience_band") \
    .show()

+---------------+-----+
|experience_band|count|
+---------------+-----+
|    Experienced|  200|
|        Fresher|   90|
|         Junior|   94|
|      Mid-Level|  116|
+---------------+-----+



In [ ]:
final_df.groupBy("interview_year", "interview_month") \
    .count() \
    .orderBy("interview_year", "interview_month") \
    .show()

+--------------+---------------+-----+
|interview_year|interview_month|count|
+--------------+---------------+-----+
|          2026|              1|   44|
|          2026|              2|   58|
|          2026|              3|   57|
|          2026|              4|   56|
|          2026|              5|   55|
|          2026|              6|   63|
|          2026|              7|   49|
|          2026|              8|   59|
|          2026|              9|   59|
+--------------+---------------+-----+



In [ ]:
!airflow version

/bin/bash: line 1: airflow: command not found


In [ ]:
!pip install -q "apache-airflow==3.4.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [ ]:
!airflow version

3.4.0


In [ ]:
!airflow config get-value core dags_folder

/root/airflow/dags


In [ ]:
dag_code = """

_IncompleteInputError: incomplete input (759594475.py, line 1)

In [ ]:
with open("/root/airflow/dags/hr_recruitment_analytics_elt.py", "w") as f:

_IncompleteInputError: incomplete input (3324071157.py, line 1)

In [ ]:
with open("/root/airflow/dags/hr_recruitment_analytics_elt.py", "w") as f:

_IncompleteInputError: incomplete input (3324071157.py, line 1)

In [ ]:
dag_code = """

_IncompleteInputError: incomplete input (759594475.py, line 1)

In [ ]:
dag_code = """

_IncompleteInputError: incomplete input (759594475.py, line 1)

In [ ]:
dag_code = """

_IncompleteInputError: incomplete input (759594475.py, line 1)

In [ ]:
Step 1: Reading source data

SyntaxError: invalid syntax (3555564354.py, line 1)

In [ ]:
Step 1: Reading source data

SyntaxError: invalid syntax (3555564354.py, line 1)

In [ ]:
import os

os.makedirs("/root/airflow/dags", exist_ok=True)

dag_code = '''
from airflow.sdk import dag, task
from datetime import datetime

@dag(
    dag_id="hr_recruitment_analytics_elt",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["HR Analytics", "PySpark", "ELT"]
)
def hr_recruitment_analytics_elt():

    @task
    def extract_source_data():
        print("Step 1: Reading source data")
        print("Source file: /content/recruitment_curated.csv")
        return "/content/recruitment_curated.csv"

    @task
    def transform_with_pyspark(source_file):
        print("Step 2: PySpark transformation")
        print("Input file:", source_file)
        return "/content/recruitment_pyspark_final.csv"

    @task
    def load_analytics_dataset(output_file):
        print("Step 3: Final analytics dataset prepared.")
        print("Output file:", output_file)

    source = extract_source_data()
    transformed = transform_with_pyspark(source)
    load_analytics_dataset(transformed)

hr_recruitment_analytics_elt()
'''

with open("/root/airflow/dags/hr_recruitment_analytics_elt.py", "w") as f:
    f.write(dag_code)

print("Airflow DAG file created successfully.")

Airflow DAG file created successfully.


In [ ]:
!airflow dag-processor --num-runs 1


Please confirm database initialize (or wait 4 seconds to skip it). Are you sure? [y/N]
2026-11-09T19:03:53.433185Z [info     ] Log template table does not exist (added in 2.3.0); skipping log template sync. [airflow.utils.db] loc=db.py:1025
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/sqlalchemy/engine/base.py", line 1977, in _exec_single_context
    self.dialect.do_execute(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        cursor, str_statement, effective_parameters, context
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/sqlalchemy/engine/default.py", line 1192, in do_execute
    cursor.execute(statement, parameters)
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
sqlite3.OperationalError: no such table: job

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/bin/airflow", line 6, in <module>
    sys.exit(main())
       

In [ ]:
!airflow db migrate

2026-11-09T19:07:27.194497Z [info     ] Performing upgrade to the metadata database [airflow.cli.commands.db_command] loc=db_command.py:134 url=sqlite:////root/airflow/airflow.db
2026-11-09T19:07:29.281157Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-11-09T19:07:29.281557Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-11-09T19:07:29.283246Z [info     ] Migrating the Airflow database [airflow.utils.db] loc=db.py:1237
2026-11-09T19:07:29.293902Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-11-09T19:07:29.294141Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-11-09T19:07:29.353635Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-11-09T19:07:29.353896Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=m

In [ ]:
monthly_analysis = (
    df_transformed
    .groupBy("interview_year", "interview_month")
    .agg(
        F.count("*").alias("total_candidates"),
        F.sum(
            F.when(F.col("candidate_outcome") == "Selected", 1).otherwise(0)
        ).alias("selected_candidates")
    )
    .withColumn(
        "selection_rate",
        F.round(
            F.col("selected_candidates") * 100.0 /
            F.col("total_candidates"),
            2
        )
    )
    .orderBy("interview_year", "interview_month")
)

monthly_analysis.show()

+--------------+---------------+----------------+-------------------+--------------+
|interview_year|interview_month|total_candidates|selected_candidates|selection_rate|
+--------------+---------------+----------------+-------------------+--------------+
|          2026|              1|              44|                 15|         34.09|
|          2026|              2|              58|                 24|         41.38|
|          2026|              3|              57|                 29|         50.88|
|          2026|              4|              56|                 27|         48.21|
|          2026|              5|              55|                 23|         41.82|
|          2026|              6|              63|                 25|         39.68|
|          2026|              7|              49|                 16|         32.65|
|          2026|              8|              59|                 28|         47.46|
|          2026|              9|              59|                

In [ ]:
output_path = "/content/recruitment_pyspark_output"

df_transformed.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_path)

print("PySpark transformed dataset saved successfully.")

PySpark transformed dataset saved successfully.


In [ ]:
import glob
import shutil

part_file = glob.glob("/content/recruitment_pyspark_output/part-*.csv")[0]

shutil.copy(
    part_file,
    "/content/recruitment_pyspark_final.csv"
)

print("Final CSV created successfully.")

Final CSV created successfully.
